In [ ]:
# Import all of the necessary libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam

In [ ]:
# Load the dataset
fed = pd.read_csv('/content/FRB_H15.csv')

# Observe first five rows
fed.head()

,Time Period,RIFSPFF_N.D
0,2025-04-24,4.33
1,2025-04-23,4.33
2,2025-04-22,4.33
3,2025-04-21,4.33
4,2025-04-20,4.33


In [ ]:
# Check data type
fed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25866 entries, 0 to 25865
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Time Period  25866 non-null  object 
 1   RIFSPFF_N.D  25866 non-null  float64
dtypes: float64(1), object(1)
memory usage: 404.3+ KB


In [ ]:
# Turn the Time Period from Object to DateTime type
fed = pd.read_csv('/content/FRB_H15.csv', parse_dates=['Time Period'], index_col='Time Period')

# Observe the first five rows again
print(fed.head())

             RIFSPFF_N.D
Time Period             
2025-04-24          4.33
2025-04-23          4.33
2025-04-22          4.33
2025-04-21          4.33
2025-04-20          4.33


In [ ]:
# Capture only the last three years of the dataset
fed = fed.head(1095)

In [ ]:
# Create a MinMaxScaler object with a feature range between 0 and 1
feds = MinMaxScaler(feature_range=(0, 1))

# Fit the scaler to the data and transform it
fedsc = feds.fit_transform(fed.values)

In [ ]:
# Create a function to predict the next day of the federal fund rate
def fed_data(fedsc, lookback=1):
    X, y = [], []
    for i in range(len(fedsc) - lookback):
        X.append(fedsc[i:(i + lookback), 0])
        y.append(fedsc[i + lookback, 0])
    return np.array(X), np.array(y)

# Set the the number of past data points
lookback = 350

# Prepare the data for the model using function
X, y = fed_data(fedsc, lookback)
# Reshape the input data to be [samples, time steps, features] for the GRU model
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

In [ ]:
# Create a Sequential model, which is a linear stack of layers
model = Sequential()
model.add(GRU(units=50, return_sequences=True, input_shape=(lookback, 1)))
model.add(GRU(units=50))
model.add(Dense(units=1))
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
model.fit(X, y, epochs=25, batch_size=32)

Epoch 1/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 18s 460ms/step - loss: 0.3100
Epoch 2/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 11s 474ms/step - loss: 0.0138
Epoch 3/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 20s 467ms/step - loss: 0.0048
Epoch 4/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 20s 467ms/step - loss: 0.0020
Epoch 5/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 20s 441ms/step - loss: 6.9749e-04
Epoch 6/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 21s 461ms/step - loss: 3.3681e-04
Epoch 7/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 11s 463ms/step - loss: 2.9297e-04
Epoch 8/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 11s 455ms/step - loss: 2.8632e-04
Epoch 9/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 11s 443ms/step - loss: 2.6273e-04
Epoch 10/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 21s 459ms/step - loss: 2.4842e-04
Epoch 11/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 20s 456ms/step - loss: 3.2501e-04
Epoch 12/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 11s 458ms/step - loss: 2.3120e-04
Epoch 13/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 20s 453ms/step - loss: 3.6141e-04
Epoch 14/25
24/24 ━━━━━━━━━━━━━━━━━━━━ 21s 472ms/step - loss: 2.2406e-04
Epo

In [ ]:
# Prepare the input data for prediction
fed_input = fedsc[-lookback:].reshape(1, lookback, 1)
fed_predict = model.predict(fed_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 500ms/step


In [ ]:
# Print the predicted Federal Fund Rate for the next day
predicted_values = feds.inverse_transform(fed_predict)
print(f"The predicted Federal Fund Rate for the next day is: {fed_predict[0][0]:.2f}%")

The predicted Federal Fund Rate for the next day is: -0.01%
